# 02 — Hybrid State Construction

**quantum-nonlocality** | Jinwon Yoo · C. Praise Anyanwu

---

This notebook covers Phase 2 of the project:

1. Initialize boundary polarization qubits in |+⟩
2. Prepare the GHZ bulk in the time-bin DOF
3. Stitch boundary qubits to the bulk via CNOT gates
4. Verify the full hybrid entangled state

This is the core construction of the paper — the moment where the two degrees of freedom become entangled into a single hybrid object.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector, DensityMatrix, partial_trace, entropy
from qiskit.visualization import plot_histogram

simulator = AerSimulator()
print("Imports OK ✓")

## 1. Qubit Register Layout

We label qubits as follows:

```
q[0]        — left boundary polarization qubit  (|+⟩_L)
q[1]        — bulk time-bin qubit 1
q[2]        — bulk time-bin qubit 2
  ⋮
q[n]        — bulk time-bin qubit n
q[n+1]      — right boundary polarization qubit (|+⟩_R)
```

Total qubits: `n + 2` where `n` is the number of bulk qubits.

In [ ]:
def build_hybrid_state(n_bulk):
    """
    Build the full hybrid entangled state:
      - q[0]        : left boundary polarization qubit |+⟩_L
      - q[1..n]     : bulk GHZ state in time-bin DOF
      - q[n+1]      : right boundary polarization qubit |+⟩_R
    
    Steps:
      1. H on q[0] and q[n+1]  → boundary qubits in |+⟩
      2. GHZ prep on q[1..n]   → bulk GHZ state
      3. CNOT(q[0], q[1])      → stitch left boundary to bulk
      4. CNOT(q[n+1], q[n])    → stitch right boundary to bulk
    """
    total = n_bulk + 2
    qc = QuantumCircuit(total)
    
    # Step 1: boundary qubits → |+⟩
    qc.h(0)           # left boundary
    qc.h(total - 1)   # right boundary
    qc.barrier()
    
    # Step 2: GHZ prep on bulk qubits q[1..n]
    qc.h(1)
    for i in range(2, n_bulk + 1):
        qc.cx(1, i)
    qc.barrier()
    
    # Step 3: CNOT stitching
    qc.cx(0, 1)               # left boundary → first bulk qubit
    qc.cx(total - 1, n_bulk)  # right boundary → last bulk qubit
    qc.barrier()
    
    return qc


# Build and draw for n=3 bulk qubits
qc_hybrid = build_hybrid_state(3)
print(f"Total qubits: {qc_hybrid.num_qubits}")
print("  q[0]   : left boundary (polarization)")
print("  q[1-3] : bulk (time-bin GHZ)")
print("  q[4]   : right boundary (polarization)")
qc_hybrid.draw('mpl')

## 2. Inspect the Hybrid Statevector

Let's look at the full statevector and identify the dominant terms.

In [ ]:
n_bulk = 2  # use n=2 for readability
qc = build_hybrid_state(n_bulk)
sv = Statevector(qc)

total_qubits = n_bulk + 2
print(f"Hybrid state statevector (n_bulk={n_bulk}, total={total_qubits} qubits):\n")
print("Non-zero amplitudes:")
for i, amp in enumerate(sv.data):
    if abs(amp) > 1e-6:
        label = format(i, f'0{total_qubits}b')
        # label format: q[0] q[1] ... q[n+1]
        print(f"  |{label}⟩  amplitude: {amp:+.4f}")

## 3. Verify Boundary Entanglement

After stitching, the boundary polarization qubits (q[0] and q[n+1]) should be entangled with each other through the bulk. We verify this by computing the reduced density matrix of just the two boundary qubits and checking its entropy.

In [ ]:
def boundary_entanglement(n_bulk):
    """Check entanglement between boundary qubits after stitching."""
    qc = build_hybrid_state(n_bulk)
    sv = Statevector(qc)
    dm = DensityMatrix(sv)
    
    total = n_bulk + 2
    # Trace out all bulk qubits — keep only q[0] and q[total-1]
    bulk_indices = list(range(1, n_bulk + 1))
    rho_boundary = partial_trace(dm, bulk_indices)
    
    S = entropy(rho_boundary, base=2)
    return S, rho_boundary


print("Boundary qubit entanglement entropy after stitching:\n")
for n in range(1, 6):
    S, _ = boundary_entanglement(n)
    print(f"  n_bulk = {n}: S(boundary) = {S:.4f} ebit")

In [ ]:
# Visualize the reduced density matrix of the boundary qubits
n_bulk = 3
S, rho_b = boundary_entanglement(n_bulk)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Real part
im0 = axes[0].imshow(np.real(rho_b.data), cmap='RdBu', vmin=-0.5, vmax=0.5)
axes[0].set_title('Re(ρ_boundary)')
axes[0].set_xticks([0,1,2,3])
axes[0].set_xticklabels(['|00⟩','|01⟩','|10⟩','|11⟩'])
axes[0].set_yticks([0,1,2,3])
axes[0].set_yticklabels(['|00⟩','|01⟩','|10⟩','|11⟩'])
plt.colorbar(im0, ax=axes[0])

# Imaginary part
im1 = axes[1].imshow(np.imag(rho_b.data), cmap='RdBu', vmin=-0.5, vmax=0.5)
axes[1].set_title('Im(ρ_boundary)')
axes[1].set_xticks([0,1,2,3])
axes[1].set_xticklabels(['|00⟩','|01⟩','|10⟩','|11⟩'])
axes[1].set_yticks([0,1,2,3])
axes[1].set_yticklabels(['|00⟩','|01⟩','|10⟩','|11⟩'])
plt.colorbar(im1, ax=axes[1])

plt.suptitle(f'Reduced density matrix of boundary qubits (n_bulk={n_bulk})\nS = {S:.4f} ebit')
plt.tight_layout()
plt.show()

## 4. Scan: Entanglement vs Bulk Size

How does the boundary entanglement change as we increase the number of bulk qubits?

In [ ]:
n_range = range(1, 8)
entropies = [boundary_entanglement(n)[0] for n in n_range]

plt.figure(figsize=(7, 4))
plt.plot(list(n_range), entropies, 'o-', lw=2, color='#534AB7')
plt.axhline(1.0, color='gray', linestyle='--', alpha=0.6, label='Maximum (1 ebit)')
plt.xlabel('Number of bulk qubits (n)')
plt.ylabel('S(boundary) [ebits]')
plt.title('Boundary entanglement entropy vs bulk size')
plt.xticks(list(n_range))
plt.ylim(0, 1.2)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Full System Measurement — Signature Check

Measure the full hybrid state. The boundary qubits should show correlated outcomes consistent with a Bell-like state, while the bulk outcomes vary.

In [ ]:
n_bulk = 2
qc_meas = build_hybrid_state(n_bulk)
qc_meas.measure_all()

compiled = transpile(qc_meas, simulator)
counts = simulator.run(compiled, shots=4096).result().get_counts()

# Extract boundary qubit correlations
# Bit string order in Qiskit: q[n+1]...q[1]q[0] (reversed)
boundary_counts = {}
total = n_bulk + 2
for bitstr, count in counts.items():
    # q[0] is the rightmost bit, q[total-1] is the leftmost
    left  = bitstr[-1]          # q[0]: left boundary
    right = bitstr[0]           # q[total-1]: right boundary
    key = f"|{left}{right}⟩"
    boundary_counts[key] = boundary_counts.get(key, 0) + count

print("Boundary qubit correlations (marginalizing over bulk):")
for k, v in sorted(boundary_counts.items()):
    print(f"  {k}: {v/4096:.3f}")

plot_histogram(counts, title=f'Full hybrid state measurements (n_bulk={n_bulk})')

## Summary

We have successfully constructed the hybrid entangled state:

$$|\Psi\rangle = \text{CNOT}_{L,R} \cdot \left( |+\rangle_L \otimes |\text{GHZ}_n\rangle_t \otimes |+\rangle_R \right)$$

Key findings:
- The CNOT stitching creates entanglement between the boundary polarization qubits through the GHZ bulk
- The boundary entanglement entropy approaches 1 ebit for larger bulk sizes
- The boundary qubits show correlated measurement outcomes — a signature that the Bell state output is within reach

**Next:** `03_mbqc_protocol.ipynb` — perform projective measurements on the bulk qubits, apply classical feedforward, and verify the deterministic Bell state output at the boundaries.